# 00 Data Exploration

Sanity-checks all local datasets through the shared Phase 1 dataloader factory.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "src").exists() else cwd.parent
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from torch.utils.data import Subset
from torchvision.utils import make_grid

from src.utils import IMAGENET_MEAN, IMAGENET_STD, get_dataloaders, set_seed

set_seed(291652)

DATA_ROOT = PROJECT_ROOT / "data"
DATASETS = ["cifar-100", "flowers-102", "stanford-cars", "tiny-imagenet-200"]
BATCH_SIZE = 64
NUM_WORKERS = 2

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root: {DATA_ROOT}")


In [ ]:
def denormalize(batch: torch.Tensor) -> torch.Tensor:
    mean = torch.tensor(IMAGENET_MEAN, dtype=batch.dtype).view(1, 3, 1, 1)
    std = torch.tensor(IMAGENET_STD, dtype=batch.dtype).view(1, 3, 1, 1)
    return (batch.cpu() * std + mean).clamp(0, 1)


def collect_targets(dataset) -> list[int]:
    if isinstance(dataset, Subset):
        base_targets = collect_targets(dataset.dataset)
        return [base_targets[index] for index in dataset.indices]
    for attribute in ("targets", "labels", "_labels"):
        if hasattr(dataset, attribute):
            return [int(value) for value in getattr(dataset, attribute)]
    if hasattr(dataset, "samples"):
        return [int(sample[1]) for sample in dataset.samples]
    raise AttributeError(f"Could not find targets for {type(dataset).__name__}")


def class_name(dataset, label: int) -> str:
    classes = getattr(dataset, "classes", None)
    if classes is not None and 0 <= label < len(classes):
        return str(classes[label])
    return str(label)


def show_sample_grid(dataset_name: str, loader) -> None:
    images, labels = next(iter(loader))
    images = images[:16]
    labels = labels[:16]
    grid = make_grid(denormalize(images), nrow=4, padding=2)

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(dataset_name)
    ax.axis("off")
    plt.show()

    label_rows = [
        {"Grid Slot": index, "Label": int(label), "Class Name": class_name(loader.dataset, int(label))}
        for index, label in enumerate(labels)
    ]
    display(pd.DataFrame(label_rows))


ORIGINAL_IMAGE_SIZES = {
    "cifar-100": "32 x 32",
    "flowers-102": "variable",
    "stanford-cars": "variable",
    "tiny-imagenet-200": "64 x 64",
}


In [ ]:
summary_rows = []
loaders_by_dataset = {}

for dataset_name in DATASETS:
    print("=" * 80)
    print(dataset_name)
    train_loader, val_loader, test_loader, num_classes = get_dataloaders(
        dataset_name=dataset_name,
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )
    loaders_by_dataset[dataset_name] = (train_loader, val_loader, test_loader, num_classes)

    train_size = len(train_loader.dataset)
    val_size = len(val_loader.dataset)
    test_size = len(test_loader.dataset)
    print(f"Classes: {num_classes}")
    print(f"Train: {train_size:,} | Val: {val_size:,} | Test: {test_size:,}")

    show_sample_grid(dataset_name, train_loader)

    train_targets = collect_targets(train_loader.dataset)
    counts = np.bincount(train_targets, minlength=num_classes)
    print(
        "Train class distribution | "
        f"min: {counts.min()} | max: {counts.max()} | mean: {counts.mean():.2f}"
    )

    summary_rows.append(
        {
            "Dataset": dataset_name,
            "Classes": num_classes,
            "Train Size": train_size,
            "Val Size": val_size,
            "Test Size": test_size,
            "Image Size (original)": ORIGINAL_IMAGE_SIZES[dataset_name],
        }
    )


In [ ]:
summary_df = pd.DataFrame(summary_rows)
display(summary_df)
